# Closing-auction volume — feature construction template (E0–E5)

**What this notebook does.** It takes the project's raw data, cleans it, and builds the model-ready feature table behind the
E0–E5 specifications shown in the midterm presentation. Run all cells; the only line you should need to change is `DATA_ROOT` in Section 0.

**Forecast being supported.** For stock *i* and trading day *t*, predict the NYSE closing-auction share volume `A(i,t)` using only
information available before the forecast is made (the close of *t − 1* for the day-ahead model).

**Output.** One row per stock-day (`date`, `permno`) with the target, eligibility flags and every feature, plus

| File | Content |
|---|---|
| `outputs/model_features/year=YYYY/…parquet` | The feature table |
| `outputs/feature_dictionary.csv` | Every feature: group (E0–E5), exact formula, raw source, when it is known |
| `outputs/calendar_features.csv` | The date-level calendar and event flags, for audit |
| `outputs/qc_report.csv` | Every data-quality and look-ahead check, with its result |
| `outputs/live_features_<date>.parquet` | Features for the next trading day after the data ends (what a live forecast would use) |

**Design rules**

1. **Reproducible from raw data.** Every feature is computed in this notebook from the raw tables or from a rule. Nothing is read from a pre-built feature file.
2. **No look-ahead.** Stock and market history use windows that end at *t − 1*. Calendar and event flags depend only on schedules published in advance. Section 10 tests this by deleting the forecast day's data and checking that no feature changes.
3. **Missing stays missing.** A missing auction print is not a zero. Features are left null where history is insufficient; imputation belongs inside the model's training folds.
4. **Small, auditable reference lists.** Dates that cannot be derived by rule (FOMC meetings, MSCI reviews, S&P 500 changes) live in short CSV files in `reference_inputs/`, each with its public source, so anyone can check or extend them.

## 0. Configuration — the only cell to edit

In [1]:
from pathlib import Path
import os

# ---- 1. Where the raw data lives (the `blackrock_data` folder of the project repo) -------------------------
DATA_ROOT = None        # e.g. Path("/Users/you/blackrock_auction_vol/blackrock_data"); None = auto-detect

# ---- 2. Where this notebook's reference lists and outputs live ----------------------------------------------
REFERENCE_DIR = Path("reference_inputs")
OUTPUT_DIR = Path("outputs")

# ---- 3. Options -----------------------------------------------------------------------------------------------
MODEL_START = "2015-01-01"          # auction targets exist from 2015; earlier CRSP rows are used as history only
MIN_AUCTION_HISTORY = 10            # a stock-day is model-ready only with >= this many auction prints in its previous 20 rows
                                    # (10 reproduces the midterm evaluation sample: 1.88M stock-days in 2020-2025)
USE_OPTIONS = True                  # add stock-level options exposure (OptionMetrics layer); needs options_daily/
USE_SP500_STOCK_LEVEL = True        # add S&P 500 membership / add / delete features
ALLOW_ESTIMATED_ANNOUNCEMENTS = False   # S&P changes whose announcement date was estimated are excluded by default
BUILD_LIVE_ROW = True               # also build features for the next trading day after the data ends
RUN_SANITY_MODEL = True             # fit the E0→E5 OLS ladder at the end as an end-to-end check
SAVE_FEATURES = True

# Environment overrides (used for automated test runs; safe to ignore)
DATA_ROOT = Path(os.environ["BLK_DATA_ROOT"]) if os.environ.get("BLK_DATA_ROOT") else DATA_ROOT
OUTPUT_DIR = Path(os.environ.get("BLK_OUTPUT_DIR", OUTPUT_DIR))

In [2]:
import json, time, warnings
from datetime import date, timedelta

import numpy as np
import pandas as pd
import duckdb          # pip install duckdb
from dateutil.easter import easter

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

if DATA_ROOT is None:      # look for a blackrock_data folder next to or above this notebook
    here = Path.cwd().resolve()
    candidates = [p / "blackrock_data" for p in [here, *here.parents]] + \
                 [p / "blackrock_auction_vol" / "blackrock_data" for p in [here, *here.parents]]
    DATA_ROOT = next((c for c in candidates if c.exists()), None)
if DATA_ROOT is None or not Path(DATA_ROOT).exists():
    raise FileNotFoundError("Set DATA_ROOT in the configuration cell to the project's blackrock_data folder.")
DATA_ROOT = Path(DATA_ROOT)

PATHS = {
    "crsp":      DATA_ROOT / "stock_daily" / "crsp_daily",       # CRSP daily stock file (WRDS crsp.dsf_v2)
    "auction":   DATA_ROOT / "stock_daily" / "auction_daily",    # NYSE closing-auction prints (WRDS TAQ)
    "options":   DATA_ROOT / "stock_daily" / "options_daily",    # OptionMetrics stock-day aggregates
    "anomalies": DATA_ROOT / "quality" / "auction_source_anomalies.parquet",
    "repo_calendar": DATA_ROOT / "events" / "event_calendar.parquet",   # used only to cross-check our calendar
    "repo_market":   DATA_ROOT / "market" / "market_daily.parquet",     # used only to cross-check our market series
}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QC = []            # every check appends a row here
FEATURES = []      # the feature dictionary is filled in as features are built

def check(name, passed, detail="", warn_only=False):
    """Record a data-quality check. Checks never silently pass: the table is shown and saved at the end.
    warn_only marks conditions that do not affect the historical table but need attention before live use."""
    status = "PASS" if bool(passed) else ("WARN" if warn_only else "FAIL")
    QC.append({"check": name, "status": status, "detail": str(detail)})
    print(f"[{status}] {name}" + (f" — {detail}" if detail != "" else ""))

def describe(name, group, formula, source, known_by):
    FEATURES.append({"feature": name, "group": group, "formula": formula, "raw_source": source, "known_by": known_by})

CAL = "Calendar: fixed by published schedule, known weeks or more ahead"
PRIOR = "Prior close (t-1)"
PREOPEN = "Pre-open of t: OCC open interest as of the close of t-1"

print("DATA_ROOT :", DATA_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())
for k in ("crsp", "auction"):
    assert PATHS[k].exists(), f"Missing required raw folder: {PATHS[k]}"
USE_OPTIONS = USE_OPTIONS and PATHS["options"].exists()
print("Options layer:", "found" if USE_OPTIONS else "not used")

DATA_ROOT : /Users/kunj/Desktop/blackrock_auction_vol/blackrock_data
OUTPUT_DIR: /Users/kunj/Desktop/blackrock_feature_template/outputs
Options layer: found


## 1. Load the raw data

| Raw table | Source and how it was extracted | Key |
|---|---|---|
| **CRSP daily** | WRDS `crsp.dsf_v2` joined to `crsp.stksecurityinfohist`. Universe: NYSE primary listing, US-incorporated ordinary common shares of corporations, regular-way, actively trading (`primaryexch='N'`, `sharetype='NS'`, `securitytype='EQTY'`, `securitysubtype='COM'`, `usincflg='Y'`, `issuertype='CORP'`, `conditionaltype='RW'`, `tradingstatusflg='A'`). No size or liquidity filter. | `date`, `permno` |
| **Auction daily** | WRDS TAQ trades. The official NYSE closing-auction print is the trade with exchange `N`, correction `00` and sale condition containing `6`; condition `M` is a repeat of the same print and is not added. Linked to CRSP `permno` with a date-specific TAQ–CRSP bridge. | `date`, `permno` |
| **Options daily** (optional) | WRDS OptionMetrics `optionm.opprcdYYYY` aggregated to one row per stock-day, linked with `wrdsapps.opcrsphist` (`score = 1`). Built by `11_options_daily.ipynb`. | `date`, `permno` |

Only the columns needed below are read.

In [3]:
CON = duckdb.connect()

def read_parquet_folder(folder, columns, filters=None):
    """Read every parquet file under a folder (year=YYYY partitions) into one DataFrame.
    DuckDB is used as the reader because it handles files written by any recent parquet writer."""
    pattern = str(Path(folder) / "**" / "*.parquet")
    n_files = len(list(Path(folder).rglob("*.parquet")))
    if n_files == 0:
        raise FileNotFoundError(f"No parquet files under {folder}")
    source = f"read_parquet('{pattern}', union_by_name=true, hive_partitioning=false)"
    where = " AND ".join(f"{col} = '{val}'" for col, _, val in (filters or [])) or "TRUE"
    total_rows = CON.execute(f"SELECT count(*) FROM {source}").fetchone()[0]
    out = CON.execute(f"SELECT {', '.join(columns)} FROM {source} WHERE {where}").df()
    out["date"] = pd.to_datetime(out["date"]).astype("datetime64[ns]")
    out["permno"] = out["permno"].astype("int64")
    return out, total_rows, n_files

def read_parquet_file(path):
    out = CON.execute(f"SELECT * FROM read_parquet('{path}')").df()
    out["date"] = pd.to_datetime(out["date"]).astype("datetime64[ns]")
    return out

t0 = time.time()
UNIVERSE_FILTER = [("primary_exchange", "=", "N"), ("security_type", "=", "EQTY"), ("security_subtype", "=", "COM"),
                   ("share_type", "=", "NS"), ("us_incorporated", "=", "Y"), ("issuer_type", "=", "CORP"),
                   ("conditional_type", "=", "RW"), ("trading_status", "=", "A")]
CRSP_COLS = ["date", "permno", "ticker", "close", "daily_volume", "daily_price_volume", "return_total", "market_cap_usd"]
crsp_raw, crsp_rows_on_disk, n = read_parquet_folder(PATHS["crsp"], CRSP_COLS, UNIVERSE_FILTER)
print(f"CRSP daily   : {len(crsp_raw):>10,} rows from {n} files  ({crsp_raw.date.min():%Y-%m-%d} to {crsp_raw.date.max():%Y-%m-%d})")

AUCTION_COLS = ["date", "permno", "auction_volume", "auction_price", "target_found"]
auction_raw, _, n = read_parquet_folder(PATHS["auction"], AUCTION_COLS)
print(f"Auction daily: {len(auction_raw):>10,} rows from {n} files  ({auction_raw.date.min():%Y-%m-%d} to {auction_raw.date.max():%Y-%m-%d})")

OPTION_ASOF_PRIOR_CLOSE = ["opt_call_oi", "opt_put_oi", "opt_total_oi", "oi_exp_today", "oi_exp_7d"]       # OI on row t = close of t-1
OPTION_SAME_DAY = ["has_options", "opt_total_vol", "opt_to_stock_vol", "iv_atm_30d", "gex_dollar"]            # known only after the close of t
if USE_OPTIONS:
    options_raw, _, n = read_parquet_folder(PATHS["options"], ["date", "permno"] + OPTION_ASOF_PRIOR_CLOSE + OPTION_SAME_DAY)
    print(f"Options daily: {len(options_raw):>10,} rows from {n} files  ({options_raw.date.min():%Y-%m-%d} to {options_raw.date.max():%Y-%m-%d})")
else:
    options_raw = None

if PATHS["anomalies"].exists():
    anomalies = read_parquet_file(PATHS["anomalies"])
    anomaly_dates = pd.to_datetime(anomalies.loc[anomalies["exclude_from_model"], "date"]).tolist()
else:   # documented fallback: the one date with an incomplete NYSE condition-6 feed
    anomaly_dates = [pd.Timestamp("2019-08-12")]
print("Dates excluded from the target (incomplete auction feed):", [f"{d:%Y-%m-%d}" for d in anomaly_dates])
print(f"Loaded in {time.time() - t0:.0f}s")

CRSP daily   :  4,927,078 rows from 15 files  (2011-01-03 to 2025-12-31)


Auction daily:  3,594,398 rows from 11 files  (2015-01-02 to 2025-12-31)


Options daily:  3,488,919 rows from 128 files  (2015-01-02 to 2025-08-29)
Dates excluded from the target (incomplete auction feed): ['2019-08-12']
Loaded in 2s


## 2. Clean and validate

Rules applied (the same rules as the project's data-treatment summary):

- **Universe** — the CRSP filter above is re-applied on load; rows failing it are dropped and counted.
- **Keys** — `(date, permno)` must be unique in every table.
- **Target** — an auction observation counts only when `target_found` is true and `auction_volume` is present. Missing prints stay missing (never zero).
- **Excluded dates** — stock-days on dates with an incomplete auction feed are not eligible targets.
- **Sanity** — no negative volumes, no non-positive auction prices, no auction larger than the day's total volume (where CRSP volume is positive).

In [4]:
check("CRSP rows passing the universe filter", True, f"{len(crsp_raw):,} of {crsp_rows_on_disk:,} rows on disk kept")
check("CRSP (date, permno) unique", not crsp_raw.duplicated(["date", "permno"]).any())
check("Auction (date, permno) unique", not auction_raw.duplicated(["date", "permno"]).any())
check("CRSP daily volume non-negative", (crsp_raw["daily_volume"].dropna() >= 0).all())
check("Auction volume positive where present", (auction_raw["auction_volume"].dropna() > 0).all())
check("Auction price positive where present", (auction_raw["auction_price"].dropna() > 0).all())
flag_consistent = (auction_raw["target_found"].fillna(False) == auction_raw["auction_volume"].notna()).all()
check("target_found agrees with auction_volume presence", flag_consistent)
if options_raw is not None:
    check("Options (date, permno) unique", not options_raw.duplicated(["date", "permno"]).any())
    check("Options volume / open interest non-negative",
          (options_raw[["opt_total_oi", "opt_total_vol", "oi_exp_today", "oi_exp_7d"]].fillna(0) >= 0).all().all())
    check("Open interest expiring within 7 days never exceeds total",
          (options_raw["oi_exp_7d"].fillna(0) <= options_raw["opt_total_oi"].fillna(0) + 1e-9).all())

def clean_inputs(crsp, auction):
    """Stock-day spine = CRSP rows. Auction prints are attached; missing prints stay missing."""
    a = auction.copy()
    a["observed_target"] = a["target_found"].fillna(False) & a["auction_volume"].notna()
    a.loc[~a["observed_target"], "auction_volume"] = np.nan
    panel = crsp.merge(a[["date", "permno", "auction_volume", "observed_target"]], on=["date", "permno"], how="left",
                       validate="one_to_one")
    panel["observed_target"] = panel["observed_target"].fillna(False).astype(bool)
    panel["eligible_target"] = panel["observed_target"] & ~panel["date"].isin(anomaly_dates)
    return panel.sort_values(["permno", "date"]).reset_index(drop=True)

panel0 = clean_inputs(crsp_raw, auction_raw)
both = panel0["observed_target"] & (panel0["daily_volume"] > 0)
check("Auction volume never exceeds CRSP daily volume", (panel0.loc[both, "auction_volume"] <= panel0.loc[both, "daily_volume"]).all())
unmatched = len(auction_raw) - auction_raw.merge(crsp_raw[["date", "permno"]], on=["date", "permno"]).shape[0]
check("Every auction row matches a CRSP stock-day", unmatched == 0, f"{unmatched} unmatched")

coverage = (panel0[panel0.date >= MODEL_START].assign(year=lambda d: d.date.dt.year)
            .groupby("year").agg(stock_days=("permno", "size"), stocks=("permno", "nunique"),
                                 observed_targets=("observed_target", "sum"), eligible_targets=("eligible_target", "sum")))
coverage["target_coverage"] = coverage["observed_targets"] / coverage["stock_days"]
coverage

[PASS] CRSP rows passing the universe filter — 4,927,078 of 4,927,078 rows on disk kept


[PASS] CRSP (date, permno) unique


[PASS] Auction (date, permno) unique
[PASS] CRSP daily volume non-negative
[PASS] Auction volume positive where present
[PASS] Auction price positive where present
[PASS] target_found agrees with auction_volume presence


[PASS] Options (date, permno) unique
[PASS] Options volume / open interest non-negative
[PASS] Open interest expiring within 7 days never exceeds total


[PASS] Auction volume never exceeds CRSP daily volume


[PASS] Every auction row matches a CRSP stock-day — 0 unmatched


,stock_days,stocks,observed_targets,eligible_targets,target_coverage
year,,,,,
2015,341909,1426,337566,337566,0.9873
2016,329950,1386,325638,325638,0.9869
2017,321370,1348,318040,318040,0.9896
2018,317656,1329,313807,313807,0.9879
2019,316550,1316,308724,308240,0.9753
2020,312434,1320,303168,303168,0.9703
2021,336769,1477,321564,321564,0.9549
2022,353277,1469,328634,328634,0.9302
2023,331648,1397,318679,318679,0.9609


## 3. Trading calendar (rule-based, so it also works for future dates)

Calendar features such as "last trading day of the month" need to know **future** trading days, so they cannot be read off historical data alone in live use.
The calendar is therefore generated from NYSE rules and then checked against the dates actually present in CRSP.

- **Holidays**: New Year's Day, Martin Luther King Jr. Day, Presidents' Day, Good Friday, Memorial Day, Juneteenth (from 2022), Independence Day, Labor Day, Thanksgiving, Christmas. A holiday on Sunday is observed on Monday; on Saturday it is observed on Friday, except New Year's Day, which is not moved back.
- **Unscheduled closures** (listed explicitly): 2012-10-29 and 2012-10-30 (Hurricane Sandy), 2018-12-05 and 2025-01-09 (national days of mourning).
- **Early closes (1:00 p.m. ET)**: the day after Thanksgiving, July 3 and December 24 when they are trading days. The closing auction still takes place on these days, at 1:00 p.m. instead of 4:00 p.m. (the TAQ auction prints are time-stamped 13:00), so they are valid target days.

In [5]:
SPECIAL_CLOSURES = [date(2012, 10, 29), date(2012, 10, 30), date(2018, 12, 5), date(2025, 1, 9)]

def nth_weekday(year, month, weekday, n):
    """n-th given weekday (Mon=0) of a month; n=-1 gives the last one."""
    if n > 0:
        d = date(year, month, 1)
        d += timedelta(days=(weekday - d.weekday()) % 7)
        return d + timedelta(weeks=n - 1)
    d = date(year, month + 1, 1) - timedelta(days=1) if month < 12 else date(year, 12, 31)
    return d - timedelta(days=(d.weekday() - weekday) % 7)

def observed(d, saturday_moves_back=True):
    if d.weekday() == 6:
        return d + timedelta(days=1)
    if d.weekday() == 5:
        return d - timedelta(days=1) if saturday_moves_back else None
    return d

def nyse_holidays(year):
    h = [observed(date(year, 1, 1), saturday_moves_back=False),
         nth_weekday(year, 1, 0, 3), nth_weekday(year, 2, 0, 3), easter(year) - timedelta(days=2),
         nth_weekday(year, 5, 0, -1), observed(date(year, 7, 4)), nth_weekday(year, 9, 0, 1),
         nth_weekday(year, 11, 3, 4), observed(date(year, 12, 25))]
    if year >= 2022:
        h.append(observed(date(year, 6, 19)))
    return {d for d in h if d is not None} | {d for d in SPECIAL_CLOSURES if d.year == year}

def trading_days(start_year, end_year):
    days = pd.bdate_range(f"{start_year}-01-01", f"{end_year}-12-31")
    holidays = set().union(*[nyse_holidays(y) for y in range(start_year, end_year + 1)])
    return pd.DatetimeIndex([d for d in days if d.date() not in holidays])

data_days = pd.DatetimeIndex(sorted(crsp_raw["date"].unique()))
first_year, last_year = data_days[0].year, data_days[-1].year
TRADING_DAYS = trading_days(first_year, last_year + 1)          # one extra year so forward-looking flags work at the data edge
rule_days = TRADING_DAYS[(TRADING_DAYS >= data_days[0]) & (TRADING_DAYS <= data_days[-1])]
only_rule, only_data = rule_days.difference(data_days), data_days.difference(rule_days)
check("Rule-based NYSE calendar equals the trading dates in CRSP", len(only_rule) == 0 and len(only_data) == 0,
      f"{len(data_days):,} trading days; rule-only {list(only_rule.strftime('%Y-%m-%d'))}, data-only {list(only_data.strftime('%Y-%m-%d'))}")

[PASS] Rule-based NYSE calendar equals the trading dates in CRSP — 3,772 trading days; rule-only [], data-only []


## 4. Reference lists (dates that cannot be derived by rule)

| File | Content | Public source | How to extend |
|---|---|---|---|
| `fomc_meetings.csv` | Every FOMC meeting 2015–2026, scheduled and unscheduled | Federal Reserve FOMC calendar pages (URL on each row); checked against those pages on 2026-11-17 | Add next year's meetings when the Fed publishes them (about a year ahead) |
| `index_event_dates.csv` | MSCI index-review effective and announcement dates; Russell reconstitution milestones; S&P quarterly rebalance dates | MSCI, FTSE Russell and S&P Dow Jones Indices schedules | Add each review when the provider announces it |
| `sp500_constituent_changes.csv` | S&P 500 additions and deletions: announcement date and the trade date (the close at which index funds trade) | Team index-rebalance dataset | Append new announcements |
| `sp500_membership_spells.csv` | S&P 500 membership intervals by `permno` | Team index-rebalance dataset | Extend the open intervals |

S&P quarterly rebalance and Russell reconstitution dates follow fixed rules, so they are **computed** below and the list is used only to confirm the rule.

In [6]:
fomc = pd.read_csv(REFERENCE_DIR / "fomc_meetings.csv", parse_dates=["meeting_start_date", "announcement_date"])
index_events = pd.read_csv(REFERENCE_DIR / "index_event_dates.csv", parse_dates=["date"])
sp_changes = pd.read_csv(REFERENCE_DIR / "sp500_constituent_changes.csv", parse_dates=["announcement_date", "trade_date"])
sp_spells = pd.read_csv(REFERENCE_DIR / "sp500_membership_spells.csv", parse_dates=["member_from", "member_to"])

# Only meetings on the published schedule that were not cancelled are usable as a day-ahead feature.
fomc_feature_dates = fomc.loc[(fomc["meeting_type"] == "scheduled") & (fomc["status"] != "cancelled"), "announcement_date"]
check("FOMC list: 8 scheduled meetings per year", (fomc[fomc.meeting_type == "scheduled"].groupby(fomc.meeting_start_date.dt.year).size() == 8).all())
check("FOMC announcement dates are trading days", fomc_feature_dates.isin(TRADING_DAYS).all())
msci_effective = index_events.loc[index_events.event_type == "msci_review_effective", "date"]
check("MSCI list: 4 reviews per year, in Feb/May/Aug/Nov",
      (msci_effective.groupby(msci_effective.dt.year).size() == 4).all() and msci_effective.dt.month.isin([2, 5, 8, 11]).all())
check("MSCI effective dates are trading days", msci_effective.isin(TRADING_DAYS).all())
REFERENCE_COVERAGE = {"fomc": fomc["announcement_date"].max(), "msci": msci_effective.max(),
                      "sp500": sp_spells["member_to"].max()}
print("Reference lists cover through:", {k: f"{v:%Y-%m-%d}" for k, v in REFERENCE_COVERAGE.items()})

[PASS] FOMC list: 8 scheduled meetings per year
[PASS] FOMC announcement dates are trading days
[PASS] MSCI list: 4 reviews per year, in Feb/May/Aug/Nov
[PASS] MSCI effective dates are trading days
Reference lists cover through: {'fomc': '2026-12-09', 'msci': '2025-11-24', 'sp500': '2025-12-31'}


## 5. Date-level calendar and event flags (E1 date part, E2, E3, E4)

All of these depend only on the calendar and on schedules published in advance, so they are known long before the forecast day.

| Flag | Rule |
|---|---|
| `sp_quarterly_rebal` | Third Friday of March, June, September, December (the S&P quarterly rebalance trades at that close). This is also the quarterly options expiration ("triple witching"), so the two cannot be separated and only one flag is kept. |
| `russell_recon` | Fourth Friday of June (annual Russell reconstitution trades at that close). |
| `msci_review` | Effective date of each MSCI index review, from the reference list. |
| `monthly_opex_ex_quarterly` | Third Friday of the other eight months (standard monthly options expiration). |
| `is_last_trading_day_month / quarter / year`, `is_first_trading_day_month` | From the trading calendar. Quarter-end days are also month-end days. |
| `is_early_close` | 1:00 p.m. close days (rule in Section 3). |
| `fomc_announcement_day` | Second day (statement day) of a scheduled FOMC meeting. Unscheduled meetings are excluded: they could not be known in advance. |

If a rule date is not a trading day, the previous trading day is used.

In [7]:
def previous_trading_day(d, days):
    d = pd.Timestamp(d)
    return d if d in days else days[days < d][-1]

def build_calendar(days):
    cal = pd.DataFrame({"date": days})
    cal["year"], cal["month"], cal["quarter"] = cal.date.dt.year, cal.date.dt.month, cal.date.dt.quarter
    ym, yq = cal.date.dt.to_period("M"), cal.date.dt.to_period("Q")
    cal["is_first_trading_day_month"] = (ym != ym.shift(1)).astype("int8")
    cal["is_last_trading_day_month"] = (ym != ym.shift(-1)).astype("int8")
    cal["is_last_trading_day_quarter"] = (yq != yq.shift(-1)).astype("int8")
    cal["is_last_trading_day_year"] = (cal.year != cal.year.shift(-1)).astype("int8")
    cal["trading_days_to_month_end"] = cal.groupby(ym).cumcount(ascending=False)

    years = range(days[0].year, days[-1].year + 1)
    third_fridays = {(y, m): previous_trading_day(nth_weekday(y, m, 4, 3), days) for y in years for m in range(1, 13)}
    quarterly = {d for (y, m), d in third_fridays.items() if m in (3, 6, 9, 12)}
    cal["monthly_opex"] = cal.date.isin(set(third_fridays.values())).astype("int8")
    cal["sp_quarterly_rebal"] = cal.date.isin(quarterly).astype("int8")            # = quarterly options expiration
    cal["monthly_opex_ex_quarterly"] = (cal.monthly_opex - cal.sp_quarterly_rebal).astype("int8")
    cal["russell_recon"] = cal.date.isin({previous_trading_day(nth_weekday(y, 6, 4, 4), days) for y in years}).astype("int8")
    cal["msci_review"] = cal.date.isin(set(msci_effective)).astype("int8")
    cal["any_index_event"] = cal[["sp_quarterly_rebal", "russell_recon", "msci_review"]].max(axis=1).astype("int8")

    early = {previous_trading_day(nth_weekday(y, 11, 3, 4) + timedelta(days=1), days) for y in years}      # day after Thanksgiving
    early |= {pd.Timestamp(y, 7, 3) for y in years} | {pd.Timestamp(y, 12, 24) for y in years}             # if they are trading days
    cal["is_early_close"] = cal.date.isin(early).astype("int8")
    cal["fomc_announcement_day"] = cal.date.isin(set(fomc_feature_dates)).astype("int8")
    # The last calendar row has no "next" row, so its end-of-period flags are not meaningful: drop the padding year's tail.
    return cal.iloc[:-1].reset_index(drop=True)

calendar = build_calendar(TRADING_DAYS)
in_data = calendar[calendar.date.isin(data_days)]

# --- cross-checks against the independent lists
for flag, etype in [("sp_quarterly_rebal", "sp_quarterly_rebalance"), ("russell_recon", "russell_reconstitution")]:
    listed = set(index_events.loc[index_events.event_type == etype, "date"])
    listed = {d for d in listed if data_days[0] <= d <= data_days[-1]}
    check(f"Rule-based {flag} dates equal the published list", set(in_data.loc[in_data[flag] == 1, "date"]) == listed, f"{len(listed)} dates")
if PATHS["repo_calendar"].exists():
    repo_cal = read_parquet_file(PATHS["repo_calendar"])
    m = in_data.merge(repo_cal, on="date", suffixes=("", "_repo"))
    pairs = [("is_early_close", "is_early_close"), ("monthly_opex", "monthly_options_expiry"), ("sp_quarterly_rebal", "quarterly_options_expiry"),
             ("is_last_trading_day_month", "is_last_trading_day_month"), ("is_last_trading_day_quarter", "is_last_trading_day_quarter"),
             ("is_last_trading_day_year", "is_last_trading_day_year"), ("is_first_trading_day_month", "is_first_trading_day_month")]
    for ours, theirs in pairs:
        theirs_col = theirs + "_repo" if theirs + "_repo" in m.columns else theirs
        diff = int((m[ours].astype(int) != m[theirs_col].astype(int)).sum())
        check(f"Calendar flag {ours} equals the repo event calendar ({theirs})", diff == 0, f"{diff} differing days of {len(m):,}")

describe("sp_quarterly_rebal", "E1", "1 on the third Friday of Mar/Jun/Sep/Dec (previous trading day if a holiday). Identical to quarterly options expiration.", "Rule on NYSE trading calendar; confirmed against S&P schedule list", CAL)
describe("russell_recon", "E1", "1 on the fourth Friday of June (annual Russell reconstitution, trades at that close).", "Rule; confirmed against FTSE Russell schedule list", CAL)
describe("msci_review", "E1", "1 on the effective date of an MSCI quarterly or semi-annual index review.", "reference_inputs/index_event_dates.csv (MSCI announcements)", CAL)
describe("any_index_event", "E1", "max(sp_quarterly_rebal, russell_recon, msci_review).", "Derived", CAL)
describe("monthly_opex_ex_quarterly", "E2", "1 on the third Friday of Jan, Feb, Apr, May, Jul, Aug, Oct, Nov (monthly options expiration that is not a quarterly expiration).", "Rule on NYSE trading calendar", CAL)
describe("is_last_trading_day_month", "E3", "1 on the last trading day of the calendar month (includes quarter- and year-end days).", "Rule-based NYSE trading calendar", CAL)
describe("is_last_trading_day_quarter", "E3", "1 on the last trading day of the calendar quarter.", "Rule-based NYSE trading calendar", CAL)
describe("is_last_trading_day_year", "E3", "1 on the last trading day of the calendar year.", "Rule-based NYSE trading calendar", CAL)
describe("is_first_trading_day_month", "E3", "1 on the first trading day of the calendar month.", "Rule-based NYSE trading calendar", CAL)
describe("is_early_close", "E3", "1 on 1:00 p.m. close days: day after Thanksgiving; July 3 and December 24 when they are trading days.", "NYSE early-close rule", CAL)
describe("fomc_announcement_day", "E4", "1 on the second day (statement day) of a scheduled FOMC meeting. Unscheduled meetings are 0 because they were not knowable in advance; the one cancelled meeting (March 2020) is 0.", "reference_inputs/fomc_meetings.csv (federalreserve.gov)", CAL)

DATE_FLAGS = ["sp_quarterly_rebal", "russell_recon", "msci_review", "any_index_event", "monthly_opex_ex_quarterly",
              "is_last_trading_day_month", "is_last_trading_day_quarter", "is_last_trading_day_year",
              "is_first_trading_day_month", "is_early_close", "fomc_announcement_day"]
in_data[in_data.date >= MODEL_START].groupby("year")[DATE_FLAGS].sum()

[PASS] Rule-based sp_quarterly_rebal dates equal the published list — 60 dates
[PASS] Rule-based russell_recon dates equal the published list — 15 dates
[PASS] Calendar flag is_early_close equals the repo event calendar (is_early_close) — 0 differing days of 3,772
[PASS] Calendar flag monthly_opex equals the repo event calendar (monthly_options_expiry) — 0 differing days of 3,772
[PASS] Calendar flag sp_quarterly_rebal equals the repo event calendar (quarterly_options_expiry) — 0 differing days of 3,772
[PASS] Calendar flag is_last_trading_day_month equals the repo event calendar (is_last_trading_day_month) — 0 differing days of 3,772
[PASS] Calendar flag is_last_trading_day_quarter equals the repo event calendar (is_last_trading_day_quarter) — 0 differing days of 3,772
[PASS] Calendar flag is_last_trading_day_year equals the repo event calendar (is_last_trading_day_year) — 0 differing days of 3,772
[PASS] Calendar flag is_first_trading_day_month equals the repo event calendar (is_firs

,sp_quarterly_rebal,russell_recon,msci_review,any_index_event,monthly_opex_ex_quarterly,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,is_early_close,fomc_announcement_day
year,,,,,,,,,,,
2015,4,1,4,9,8,12,4,1,12,2,8
2016,4,1,4,9,8,12,4,1,12,1,8
2017,4,1,4,9,8,12,4,1,12,2,8
2018,4,1,4,9,8,12,4,1,12,3,8
2019,4,1,4,9,8,12,4,1,12,3,8
2020,4,1,4,9,8,12,4,1,12,2,7
2021,4,1,4,9,8,12,4,1,12,1,8
2022,4,1,4,9,8,12,4,1,12,1,8
2023,4,1,4,9,8,12,4,1,12,2,8


## 6. E0 — historical auction-volume reference and the prediction target

For each stock, using its own rows in date order:

- `median20_auction_volume` $= M_{20}(i,t)$ = median of the auction volumes observed on the stock's **previous 20 trading rows** (rows *t − 20 … t − 1*; missing prints are skipped; at least one observation is required).
- `n_obs_median20` = how many of those 20 rows had an auction print.
- **Target** `y_log_ratio` $= \log(1 + A(i,t)) - \log(1 + M_{20}(i,t))$.

E0 is the forecast $\hat{A} = M_{20}$ (plus a fitted constant in log space). The same definition is used by the project's daily panel (`median20_auction_volume`).

A stock-day is flagged `model_ready` when its target is eligible **and** `n_obs_median20 >= MIN_AUCTION_HISTORY` (10 by default), so the reference is not based on a handful of prints.

In [8]:
def prior_window(panel, column, window, how, min_periods=1):
    """Statistic of `column` over the stock's previous `window` rows, i.e. rows t-window ... t-1 (never row t)."""
    shifted = panel.groupby("permno", sort=False)[column].shift(1)
    rolled = shifted.groupby(panel["permno"], sort=False).rolling(window, min_periods=min_periods)
    return getattr(rolled, how)().reset_index(level=0, drop=True)

def add_e0(panel):
    panel["median20_auction_volume"] = prior_window(panel, "auction_volume", 20, "median")
    panel["n_obs_median20"] = prior_window(panel, "auction_volume", 20, "count").fillna(0).astype("int16")
    panel["log_median20_auction_volume"] = np.log1p(panel["median20_auction_volume"])
    panel["y_log_ratio"] = np.log1p(panel["auction_volume"]) - panel["log_median20_auction_volume"]
    return panel

describe("median20_auction_volume", "E0", "Median of auction_volume over the stock's previous 20 rows (t-20..t-1), skipping missing prints; needs >= 1 observation.", "TAQ closing-auction prints", PRIOR)
describe("log_median20_auction_volume", "E0", "log(1 + median20_auction_volume). The model's offset: forecast = exp(offset + fitted value) - 1.", "TAQ closing-auction prints", PRIOR)
describe("n_obs_median20", "E0", "Number of non-missing auction prints among the stock's previous 20 rows.", "TAQ closing-auction prints", PRIOR)

## 7. E1 stock-level part — S&P 500 membership and announced changes (optional)

| Feature | Definition |
|---|---|
| `sp500_member_lag1` | 1 if the stock was an S&P 500 member on the previous trading day. |
| `sp500_member_x_sp_rebal` | `sp500_member_lag1 × sp_quarterly_rebal` (index funds trade members at the quarterly rebalance close). |
| `sp500_add_pending`, `sp500_del_pending` | 1 from the trading day **after** the announcement through the trade date (announcements are made after the close). |
| `sp500_add_trade_day`, `sp500_del_trade_day` | 1 on the trade date itself, only if the change was announced before that day. |
| `td_to_sp500_change` | Trading days until the trade date while a change is pending (0 on the trade date); missing otherwise. |

Changes whose announcement date is missing or was estimated are **not used** unless `ALLOW_ESTIMATED_ANNOUNCEMENTS = True`, because an estimated date could reveal a change before it was public.

In [9]:
SP_FEATURES = ["sp500_member_lag1", "sp500_member_x_sp_rebal", "sp500_add_pending", "sp500_del_pending",
               "sp500_add_trade_day", "sp500_del_trade_day", "td_to_sp500_change"]

def add_sp500(panel, days):
    day_pos = pd.Series(np.arange(len(days)), index=days)
    prev_day = pd.Series(days[:-1], index=days[1:])                         # trading day -> previous trading day
    keys = panel[["permno", "date"]].copy()
    keys["prev_date"] = keys["date"].map(prev_day)
    spans = keys.merge(sp_spells, on="permno", how="inner")
    member = spans[(spans.prev_date >= spans.member_from) & (spans.prev_date <= spans.member_to)][["permno", "date"]].drop_duplicates()
    member["sp500_member_lag1"] = 1
    panel = panel.merge(member, on=["permno", "date"], how="left")
    panel["sp500_member_lag1"] = panel["sp500_member_lag1"].fillna(0).astype("int8")

    usable = sp_changes.dropna(subset=["announcement_date"])
    if not ALLOW_ESTIMATED_ANNOUNCEMENTS:
        usable = usable[usable["announcement_date_estimated"] == 0]
    usable = usable[(usable["announcement_date"] < usable["trade_date"]) & usable["trade_date"].isin(days)]   # public before the trade day
    rows = []
    for ev in usable.itertuples():
        window = days[(days > ev.announcement_date) & (days <= ev.trade_date)]
        kind = "add" if ev.event == "add" else "del"
        for d in window:
            rows.append((ev.permno, d, kind, int(day_pos[ev.trade_date] - day_pos[d])))
    pend = pd.DataFrame(rows, columns=["permno", "date", "kind", "td"])
    for kind in ("add", "del"):
        p = pend[pend.kind == kind].groupby(["permno", "date"], as_index=False)["td"].min()
        p[f"sp500_{kind}_pending"] = 1
        p[f"sp500_{kind}_trade_day"] = (p["td"] == 0).astype(int)
        panel = panel.merge(p.rename(columns={"td": f"td_{kind}"}), on=["permno", "date"], how="left")
        for c in (f"sp500_{kind}_pending", f"sp500_{kind}_trade_day"):
            panel[c] = panel[c].fillna(0).astype("int8")
    panel["td_to_sp500_change"] = panel[["td_add", "td_del"]].min(axis=1)
    panel = panel.drop(columns=["td_add", "td_del"])
    return panel, len(usable), len(sp_changes)

if USE_SP500_STOCK_LEVEL:
    describe("sp500_member_lag1", "E1", "1 if the stock was in the S&P 500 on the previous trading day.", "reference_inputs/sp500_membership_spells.csv", PRIOR)
    describe("sp500_member_x_sp_rebal", "E1", "sp500_member_lag1 * sp_quarterly_rebal.", "Derived", PRIOR)
    describe("sp500_add_pending", "E1", "1 from the trading day after an S&P 500 addition is announced through its trade date.", "reference_inputs/sp500_constituent_changes.csv", PRIOR)
    describe("sp500_del_pending", "E1", "1 from the trading day after an S&P 500 deletion is announced through its trade date.", "reference_inputs/sp500_constituent_changes.csv", PRIOR)
    describe("sp500_add_trade_day", "E1", "1 on the trade date of an announced S&P 500 addition (index funds buy at that close).", "reference_inputs/sp500_constituent_changes.csv", PRIOR)
    describe("sp500_del_trade_day", "E1", "1 on the trade date of an announced S&P 500 deletion.", "reference_inputs/sp500_constituent_changes.csv", PRIOR)
    describe("td_to_sp500_change", "E1", "Trading days from t to the trade date while an addition or deletion is pending; missing when nothing is pending.", "reference_inputs/sp500_constituent_changes.csv", PRIOR)

## 8. E2 extension — stock-level options exposure (optional)

The calendar flag in E2 says *whether* options expire; the OptionMetrics layer says *how much* is expiring for each stock.

**Timing.** OptionMetrics reports open interest with a one-day lag: the value on row *t* is the open interest at the close of *t − 1*
(confirmed in our data: daily changes in open interest line up with the previous day's volume, correlation 0.91, not the same day's, 0.65).
So open-interest columns on row *t* are usable for a forecast of day *t*; in live use they come from the OCC's morning open-interest file.
Volume, implied volatility and gamma on row *t* are only known after the close of *t*, so they are lagged one stock row.

| Feature | Definition |
|---|---|
| `opt_oi_exp_today_share` | Open interest expiring on *t* ÷ total open interest. |
| `opt_oi_exp_7d_share` | Open interest expiring within 7 calendar days ÷ total open interest. |
| `opt_log_total_oi` | log(1 + total open interest, in contracts). |
| `opt_pc_oi_ratio` | Put open interest ÷ call open interest. |
| `opt_has_options_lag1` | 1 if the stock had listed options on its previous row. |
| `opt_to_stock_vol_lag1` | Previous row's option volume × 100 ÷ stock volume. |
| `opt_iv_atm_30d_lag1` | Previous row's 30-day at-the-money implied volatility. |
| `opt_abs_gex_to_dollar_volume_lag1` | Previous row's \|dealer gamma exposure in dollars per 1% move\| ÷ 20-row average dollar volume. Gamma exposure assumes dealers are long calls and short puts, so only its size is used. |

Stocks without listed options keep missing values (not zero) and `opt_has_options_lag1 = 0`.

In [10]:
OPT_FEATURES = ["opt_oi_exp_today_share", "opt_oi_exp_7d_share", "opt_log_total_oi", "opt_pc_oi_ratio",
                "opt_has_options_lag1", "opt_to_stock_vol_lag1", "opt_iv_atm_30d_lag1", "opt_abs_gex_to_dollar_volume_lag1"]

def add_options(panel, options):
    panel = panel.merge(options, on=["date", "permno"], how="left", validate="one_to_one")
    total = panel["opt_total_oi"].where(panel["opt_total_oi"] > 0)
    panel["opt_oi_exp_today_share"] = panel["oi_exp_today"] / total
    panel["opt_oi_exp_7d_share"] = panel["oi_exp_7d"] / total
    panel["opt_log_total_oi"] = np.log1p(panel["opt_total_oi"])
    panel["opt_pc_oi_ratio"] = panel["opt_put_oi"] / panel["opt_call_oi"].where(panel["opt_call_oi"] > 0)
    lagged = panel.groupby("permno", sort=False)[OPTION_SAME_DAY].shift(1)
    panel["opt_has_options_lag1"] = lagged["has_options"]
    panel["opt_to_stock_vol_lag1"] = lagged["opt_to_stock_vol"]
    panel["opt_iv_atm_30d_lag1"] = lagged["iv_atm_30d"]
    avg20_dollar = prior_window(panel, "daily_price_volume", 20, "mean")
    panel["opt_abs_gex_to_dollar_volume_lag1"] = lagged["gex_dollar"].abs() / avg20_dollar.where(avg20_dollar > 0)
    return panel.drop(columns=OPTION_ASOF_PRIOR_CLOSE + OPTION_SAME_DAY)

if USE_OPTIONS:
    OM = "OptionMetrics IvyDB (optionm.opprcd), stock-day aggregates"
    describe("opt_oi_exp_today_share", "E2+", "oi_exp_today / opt_total_oi, where oi_exp_today is open interest in contracts expiring on t.", OM, PREOPEN)
    describe("opt_oi_exp_7d_share", "E2+", "oi_exp_7d / opt_total_oi, where oi_exp_7d is open interest expiring in [t, t+7 calendar days].", OM, PREOPEN)
    describe("opt_log_total_oi", "E2+", "log(1 + opt_total_oi), total call + put open interest in contracts.", OM, PREOPEN)
    describe("opt_pc_oi_ratio", "E2+", "opt_put_oi / opt_call_oi.", OM, PREOPEN)
    describe("opt_has_options_lag1", "E2+", "1 if the stock had any listed option series on its previous row, 0 if not, missing where the options layer has no data.", OM, PRIOR)
    describe("opt_to_stock_vol_lag1", "E2+", "Previous row's (option contracts traded * 100) / CRSP share volume.", OM + "; CRSP", PRIOR)
    describe("opt_iv_atm_30d_lag1", "E2+", "Previous row's mean of the 30-day, 50-delta call and put implied volatilities (optionm.vsurfd).", "OptionMetrics volatility surface", PRIOR)
    describe("opt_abs_gex_to_dollar_volume_lag1", "E2+", "abs(previous row's gex_dollar) / mean dollar volume over the previous 20 rows; gex_dollar = sum(gamma * OI * contract size) * price^2 * 0.01, calls minus puts.", OM + "; CRSP", PRIOR)

## 9. E5 — lagged volatility and trading-volume regimes, with event interactions

**Stock level** (own previous rows only)

- `volatility20` = sample standard deviation of `return_total` over the previous 20 rows (needs ≥ 2).
- `abnormal_volume_lag1` = previous row's share volume ÷ mean share volume over the previous 20 rows.
- `size_pct_lag1` = cross-sectional percentile rank (0–1) of the previous row's market capitalisation among stocks trading on *t*.

**Market level** (computed here from the CRSP cross-section, then lagged)

- `market_return_vw(t)` = Σ return × market cap ÷ Σ market cap across the universe on day *t*.
- `market_volatility20` = standard deviation of `market_return_vw` over the previous 20 trading days.
- `market_abnormal_volume_lag1` = previous day's total share volume ÷ its mean over the previous 20 trading days.

**Regimes** (binary, thresholds use past data only)

- `high_vol_regime` = 1 if `market_volatility20` is above its own median over the trailing 252 trading days.
- `high_volume_regime` = 1 if `market_abnormal_volume_lag1` > 1.

**Interactions**: `index_event_x_high_vol`, `opex_x_high_vol`, `month_end_x_high_vol`, and `russell_recon_x_small_cap = russell_recon × (1 − size_pct_lag1)`.

These follow the E5 description on the methodology slide ("lagged volatility and trading-volume regimes, including event interactions"); the thresholds are simple, documented choices that can be changed here.

In [11]:
def market_series(crsp):
    """Daily market aggregates from the CRSP cross-section, then lagged so day t only sees days <= t-1."""
    ok = crsp["return_total"].notna() & crsp["market_cap_usd"].notna()
    weighted = (crsp["return_total"] * crsp["market_cap_usd"]).where(ok)
    mkt = pd.DataFrame({
        "market_return_vw": weighted.groupby(crsp["date"]).sum() / crsp["market_cap_usd"].where(ok).groupby(crsp["date"]).sum(),
        "market_total_volume": crsp.groupby("date")["daily_volume"].sum(min_count=1),
    }).sort_index()
    return mkt

def add_market_regimes(calendar_frame, mkt):
    m = mkt.reindex(calendar_frame["date"])                  # future / live dates have no same-day data, which is fine: only lags are used
    out = pd.DataFrame({"date": calendar_frame["date"].values})
    out["market_volatility20"] = m["market_return_vw"].shift(1).rolling(20, min_periods=2).std().values
    out["market_abnormal_volume_lag1"] = (m["market_total_volume"].shift(1) / m["market_total_volume"].shift(1).rolling(20, min_periods=1).mean()).values
    trailing_median = out["market_volatility20"].rolling(252, min_periods=126).median()
    out["high_vol_regime"] = (out["market_volatility20"] > trailing_median).astype("float").where(trailing_median.notna())
    out["high_volume_regime"] = (out["market_abnormal_volume_lag1"] > 1).astype("float").where(out["market_abnormal_volume_lag1"].notna())
    return out

def add_e5_stock(panel):
    panel["volatility20"] = prior_window(panel, "return_total", 20, "std", min_periods=2)
    avg20_volume = prior_window(panel, "daily_volume", 20, "mean")
    lag1_volume = panel.groupby("permno", sort=False)["daily_volume"].shift(1)
    panel["abnormal_volume_lag1"] = lag1_volume / avg20_volume.where(avg20_volume > 0)
    lag1_cap = panel.groupby("permno", sort=False)["market_cap_usd"].shift(1)
    panel["size_pct_lag1"] = lag1_cap.groupby(panel["date"]).rank(pct=True)
    return panel

def add_interactions(panel):
    panel["index_event_x_high_vol"] = panel["any_index_event"] * panel["high_vol_regime"]
    panel["opex_x_high_vol"] = panel["monthly_opex_ex_quarterly"] * panel["high_vol_regime"]
    panel["month_end_x_high_vol"] = panel["is_last_trading_day_month"] * panel["high_vol_regime"]
    panel["russell_recon_x_small_cap"] = panel["russell_recon"] * (1 - panel["size_pct_lag1"])
    return panel

describe("volatility20", "E5", "Sample std of return_total over the stock's previous 20 rows (needs >= 2).", "CRSP daily returns", PRIOR)
describe("abnormal_volume_lag1", "E5", "daily_volume on the previous row / mean daily_volume over the previous 20 rows.", "CRSP daily volume", PRIOR)
describe("size_pct_lag1", "E5", "Percentile rank (0-1) across stocks on date t of the previous row's market_cap_usd.", "CRSP market capitalisation", PRIOR)
describe("market_volatility20", "E5", "Std over the previous 20 trading days of market_return_vw = sum(return_total * market_cap) / sum(market_cap) across the universe.", "CRSP cross-section", PRIOR)
describe("market_abnormal_volume_lag1", "E5", "Previous day's total universe share volume / its mean over the previous 20 trading days.", "CRSP cross-section", PRIOR)
describe("high_vol_regime", "E5", "1 if market_volatility20 > its median over the trailing 252 trading days (needs >= 126).", "Derived", PRIOR)
describe("high_volume_regime", "E5", "1 if market_abnormal_volume_lag1 > 1.", "Derived", PRIOR)
describe("index_event_x_high_vol", "E5", "any_index_event * high_vol_regime.", "Derived", PRIOR)
describe("opex_x_high_vol", "E5", "monthly_opex_ex_quarterly * high_vol_regime.", "Derived", PRIOR)
describe("month_end_x_high_vol", "E5", "is_last_trading_day_month * high_vol_regime.", "Derived", PRIOR)
describe("russell_recon_x_small_cap", "E5", "russell_recon * (1 - size_pct_lag1).", "Derived", PRIOR)
E5_FEATURES = ["volatility20", "abnormal_volume_lag1", "size_pct_lag1", "market_volatility20", "market_abnormal_volume_lag1",
               "high_vol_regime", "high_volume_regime", "index_event_x_high_vol", "opex_x_high_vol", "month_end_x_high_vol",
               "russell_recon_x_small_cap"]

## 10. Run the pipeline

`build_features` chains the steps above. It is a function so that the exact same code can be re-run on truncated data for the look-ahead test and for the live row.

In [12]:
E0_FEATURES = ["median20_auction_volume", "log_median20_auction_volume", "n_obs_median20"]
ID_COLUMNS = ["date", "permno", "ticker"]
TARGET_COLUMNS = ["auction_volume", "observed_target", "eligible_target", "y_log_ratio", "model_ready"]

def build_features(crsp, auction, options, calendar_frame, verbose=True):
    t = time.time()
    panel = clean_inputs(crsp, auction)
    panel = add_e0(panel)
    panel = add_e5_stock(panel)
    if USE_OPTIONS and options is not None:
        panel = add_options(panel, options)
    cal_cols = calendar_frame[["date"] + DATE_FLAGS].merge(add_market_regimes(calendar_frame, market_series(crsp)), on="date")
    panel = panel.merge(cal_cols, on="date", how="left", validate="many_to_one")
    if USE_SP500_STOCK_LEVEL:
        panel, used, listed = add_sp500(panel, pd.DatetimeIndex(calendar_frame["date"]))
        panel["sp500_member_x_sp_rebal"] = (panel["sp500_member_lag1"] * panel["sp_quarterly_rebal"]).astype("int8")
        if verbose:
            print(f"S&P 500 changes used: {used} of {listed} (the rest have a missing, estimated or same-day announcement date)")
    panel = add_interactions(panel)
    panel["model_ready"] = panel["eligible_target"] & (panel["n_obs_median20"] >= MIN_AUCTION_HISTORY)
    if verbose:
        print(f"Built {panel.shape[0]:,} stock-day rows x {panel.shape[1]} columns in {time.time() - t:.0f}s")
    return panel

FEATURE_COLUMNS = (E0_FEATURES + DATE_FLAGS + (SP_FEATURES if USE_SP500_STOCK_LEVEL else [])
                   + (OPT_FEATURES if USE_OPTIONS else []) + E5_FEATURES)
full = build_features(crsp_raw, auction_raw, options_raw, calendar)
features = full.loc[full["date"] >= MODEL_START, ID_COLUMNS + TARGET_COLUMNS + FEATURE_COLUMNS].reset_index(drop=True)
print(f"Feature table: {len(features):,} rows, {len(FEATURE_COLUMNS)} features, {int(features.model_ready.sum()):,} model-ready rows")
features.tail(3).T

S&P 500 changes used: 306 of 358 (the rest have a missing, estimated or same-day announcement date)
Built 4,927,078 stock-day rows x 53 columns in 24s


Feature table: 3,594,404 rows, 40 features, 3,466,653 model-ready rows


,3594401,3594402,3594403
date,2025-12-29 00:00:00,2025-12-30 00:00:00,2025-12-31 00:00:00
permno,93426,93426,93426
ticker,VPG,VPG,VPG
auction_volume,10829,6102,11548
observed_target,True,True,True
eligible_target,True,True,True
y_log_ratio,-0.0143,-0.5878,0.0500
model_ready,True,True,True
median20_auction_volume,"10,985.0000","10,985.0000","10,985.0000"
log_median20_auction_volume,9.3044,9.3044,9.3044


## 11. Checks on the output

**Look-ahead test.** Take one forecast day *T* (an index-event day). Rebuild every feature twice from the two years of raw data ending at *T*: once as is, and once
after **deleting everything that is only known at or after the close of *T*** (that day's prices, volumes, returns, market caps, auction prints, and same-day options fields).
If any feature for day *T* changes, it was using information from day *T*. All features must be identical.

In [13]:
def lookahead_test(T):
    T = pd.Timestamp(T)
    start = T - pd.Timedelta(days=760)
    c = crsp_raw[(crsp_raw.date >= start) & (crsp_raw.date <= T)].copy()
    a = auction_raw[(auction_raw.date >= start) & (auction_raw.date <= T)].copy()
    o = options_raw[(options_raw.date >= start) & (options_raw.date <= T)].copy() if options_raw is not None else None
    cal = calendar[calendar.date >= c.date.min()].reset_index(drop=True)
    with_T = build_features(c, a, o, cal, verbose=False)
    c.loc[c.date == T, ["close", "daily_volume", "daily_price_volume", "return_total", "market_cap_usd"]] = np.nan
    a = a[a.date < T]
    if o is not None:
        o.loc[o.date == T, OPTION_SAME_DAY] = np.nan
    without_T = build_features(c, a, o, cal, verbose=False)
    x = with_T.loc[with_T.date == T].set_index("permno")[FEATURE_COLUMNS].astype("float64")
    y = without_T.loc[without_T.date == T].set_index("permno")[FEATURE_COLUMNS].astype("float64").reindex(x.index)
    same = np.isclose(x, y, equal_nan=True, rtol=1e-10, atol=1e-12)
    changed = [col for col, okay in zip(FEATURE_COLUMNS, same.all(axis=0)) if not okay]
    return len(x), changed

for T in ["2023-03-17", "2024-06-28", "2025-05-30"]:      # S&P quarterly rebalance, Russell reconstitution, MSCI review
    n_stocks, changed = lookahead_test(T)
    check(f"Look-ahead test on {T}: no feature uses day-T data", len(changed) == 0, f"{n_stocks:,} stocks, {len(FEATURE_COLUMNS)} features; changed: {changed or 'none'}")

[PASS] Look-ahead test on 2023-03-17: no feature uses day-T data — 1,339 stocks, 40 features; changed: none


[PASS] Look-ahead test on 2024-06-28: no feature uses day-T data — 1,278 stocks, 40 features; changed: none


[PASS] Look-ahead test on 2025-05-30: no feature uses day-T data — 1,241 stocks, 40 features; changed: none


In [14]:
check("Feature table (date, permno) unique", not features.duplicated(["date", "permno"]).any())
documented = {f["feature"] for f in FEATURES}
check("Every feature has a dictionary entry", set(FEATURE_COLUMNS) <= documented, sorted(set(FEATURE_COLUMNS) - documented))
check("Target never missing on model-ready rows", features.loc[features.model_ready, "y_log_ratio"].notna().all())
check("Binary flags only take values 0/1", all(set(features[c].dropna().unique()) <= {0, 1} for c in DATE_FLAGS))

# Independent recomputation of the E0 reference for a random sample of stocks (plain Python loop, no rolling helper)
rng = np.random.default_rng(7)
sample = rng.choice(features.permno.unique(), 25, replace=False)
bad = 0
for p in sample:
    s = full.loc[full.permno == p, ["date", "auction_volume", "median20_auction_volume"]].reset_index(drop=True)
    for i in rng.choice(len(s), min(40, len(s)), replace=False):
        prior = s.auction_volume.iloc[max(0, i - 20):i].dropna()
        expected = np.median(prior) if len(prior) else np.nan
        bad += not np.isclose(s.median20_auction_volume.iloc[i], expected, equal_nan=True)
check("median20_auction_volume matches an independent recomputation", bad == 0, f"{bad} mismatches in 1,000 sampled stock-days")

if PATHS["repo_market"].exists():      # our market series vs the repo's market_daily table
    repo_m = read_parquet_file(PATHS["repo_market"]).set_index("date")
    ours = market_series(crsp_raw)
    j = ours.join(repo_m[["market_return_vw", "market_total_volume"]], rsuffix="_repo").dropna()
    check("Recomputed market value-weighted return equals the repo's market_daily", np.allclose(j.market_return_vw, j.market_return_vw_repo, atol=1e-9),
          f"max abs diff {np.abs(j.market_return_vw - j.market_return_vw_repo).max():.2e} over {len(j):,} days")
    check("Recomputed market total volume equals the repo's market_daily", np.allclose(j.market_total_volume, j.market_total_volume_repo, rtol=1e-9))

mr = features[features.model_ready]
summary = pd.DataFrame({
    "missing_pct_model_ready": (mr[FEATURE_COLUMNS].isna().mean() * 100).round(3),
    "mean": mr[FEATURE_COLUMNS].astype("float64").mean(),
    "min": mr[FEATURE_COLUMNS].astype("float64").min(),
    "max": mr[FEATURE_COLUMNS].astype("float64").max(),
})
summary

[PASS] Feature table (date, permno) unique
[PASS] Every feature has a dictionary entry — []
[PASS] Target never missing on model-ready rows
[PASS] Binary flags only take values 0/1


[PASS] median20_auction_volume matches an independent recomputation — 0 mismatches in 1,000 sampled stock-days


[PASS] Recomputed market value-weighted return equals the repo's market_daily — max abs diff 1.39e-17 over 3,772 days
[PASS] Recomputed market total volume equals the repo's market_daily


,missing_pct_model_ready,mean,min,max
median20_auction_volume,0.0000,"142,198.8423",1.0000,"9,465,336.5000"
log_median20_auction_volume,0.0000,10.7407,0.6931,16.0631
n_obs_median20,0.0000,19.9128,10.0000,20.0000
sp_quarterly_rebal,0.0000,0.0160,0.0000,1.0000
russell_recon,0.0000,0.0040,0.0000,1.0000
msci_review,0.0000,0.0160,0.0000,1.0000
any_index_event,0.0000,0.0360,0.0000,1.0000
monthly_opex_ex_quarterly,0.0000,0.0320,0.0000,1.0000
is_last_trading_day_month,0.0000,0.0480,0.0000,1.0000
is_last_trading_day_quarter,0.0000,0.0160,0.0000,1.0000


## 12. Save

In [15]:
dictionary = pd.DataFrame(FEATURES)
dictionary = dictionary[dictionary.feature.isin(FEATURE_COLUMNS)].set_index("feature").loc[FEATURE_COLUMNS].reset_index()
dictionary.to_csv(OUTPUT_DIR / "feature_dictionary.csv", index=False)
calendar[calendar.date.isin(data_days) & (calendar.date >= MODEL_START)][["date"] + DATE_FLAGS].to_csv(OUTPUT_DIR / "calendar_features.csv", index=False)

if SAVE_FEATURES:
    out_root = OUTPUT_DIR / "model_features"
    for year, chunk in features.groupby(features.date.dt.year):
        folder = out_root / f"year={year}"; folder.mkdir(parents=True, exist_ok=True)
        chunk.to_parquet(folder / f"model_features_{year}.parquet", index=False, compression="zstd")
    size_mb = sum(f.stat().st_size for f in out_root.rglob("*.parquet")) / 1e6
    print(f"Saved {len(features):,} rows to {out_root} ({size_mb:.0f} MB)")
print(dictionary.groupby("group").size().rename("features").to_frame().T)
dictionary

Saved 3,594,404 rows to outputs/model_features (276 MB)
group     E0  E1  E2  E2+  E3  E4  E5
features   3  11   1    8   5   1  11


,feature,group,formula,raw_source,known_by
0,median20_auction_volume,E0,Median of auction_volume over the stock's prev...,TAQ closing-auction prints,Prior close (t-1)
1,log_median20_auction_volume,E0,log(1 + median20_auction_volume). The model's ...,TAQ closing-auction prints,Prior close (t-1)
2,n_obs_median20,E0,Number of non-missing auction prints among the...,TAQ closing-auction prints,Prior close (t-1)
3,sp_quarterly_rebal,E1,1 on the third Friday of Mar/Jun/Sep/Dec (prev...,Rule on NYSE trading calendar; confirmed again...,"Calendar: fixed by published schedule, known w..."
4,russell_recon,E1,1 on the fourth Friday of June (annual Russell...,Rule; confirmed against FTSE Russell schedule ...,"Calendar: fixed by published schedule, known w..."
5,msci_review,E1,1 on the effective date of an MSCI quarterly o...,reference_inputs/index_event_dates.csv (MSCI a...,"Calendar: fixed by published schedule, known w..."
6,any_index_event,E1,"max(sp_quarterly_rebal, russell_recon, msci_re...",Derived,"Calendar: fixed by published schedule, known w..."
7,monthly_opex_ex_quarterly,E2,"1 on the third Friday of Jan, Feb, Apr, May, J...",Rule on NYSE trading calendar,"Calendar: fixed by published schedule, known w..."
8,is_last_trading_day_month,E3,1 on the last trading day of the calendar mont...,Rule-based NYSE trading calendar,"Calendar: fixed by published schedule, known w..."
9,is_last_trading_day_quarter,E3,1 on the last trading day of the calendar quar...,Rule-based NYSE trading calendar,"Calendar: fixed by published schedule, known w..."


## 13. Live row — features for the next trading day

This is the deployment view: standing after the close of the last day in the data, what would the feature row for the **next** trading day look like?
The next day is appended with every same-day value missing and the same `build_features` code is run.

What a live run needs, by feature type:

- **Calendar and event flags** — nothing beyond the reference lists (they must cover the forecast date).
- **Prior-close features** — CRSP-equivalent prices, volumes, returns and market caps through *t − 1*, and auction prints through *t − 1*.
- **Open-interest features** — the OCC open interest as of the close of *t − 1* (published before the open of *t*). They are missing in this demonstration row because the historical file for day *t* is not available before day *t* ends.

In [16]:
if BUILD_LIVE_ROW:
    last_day = data_days[-1]
    live_day = TRADING_DAYS[TRADING_DAYS > last_day][0]
    recent = crsp_raw[crsp_raw.date >= last_day - pd.Timedelta(days=760)]
    stub = recent.loc[recent.date == last_day, ["permno", "ticker"]].assign(date=live_day)
    live_crsp = pd.concat([recent, stub], ignore_index=True)
    live_opts = options_raw[options_raw.date >= last_day - pd.Timedelta(days=760)] if options_raw is not None else None
    live_cal = calendar[calendar.date >= live_crsp.date.min()].reset_index(drop=True)
    live = build_features(live_crsp, auction_raw[auction_raw.date >= live_crsp.date.min()], live_opts, live_cal, verbose=False)
    live = live.loc[live.date == live_day, ID_COLUMNS + FEATURE_COLUMNS].reset_index(drop=True)
    for name, covered in REFERENCE_COVERAGE.items():
        needed = last_day if name == "sp500" else live_day          # membership is used with a one-day lag
        check(f"Reference list '{name}' covers the live date {live_day:%Y-%m-%d}", covered >= needed,
              f"list ends {covered:%Y-%m-%d}" + ("; extend reference_inputs before relying on this flag in live use" if covered < needed else ""),
              warn_only=True)
    if options_raw is not None and options_raw.date.max() < last_day:
        print(f"Note: the options layer ends {options_raw.date.max():%Y-%m-%d}, so options features are missing after that date.")
    live.to_parquet(OUTPUT_DIR / f"live_features_{live_day:%Y%m%d}.parquet", index=False)
    print(f"Live feature row for {live_day:%Y-%m-%d}: {len(live):,} stocks")
    display(pd.DataFrame({"available_pct": (live[FEATURE_COLUMNS].notna().mean() * 100).round(1)}).T)

[PASS] Reference list 'fomc' covers the live date 2026-01-02 — list ends 2026-12-09
[WARN] Reference list 'msci' covers the live date 2026-01-02 — list ends 2025-11-24; extend reference_inputs before relying on this flag in live use
[PASS] Reference list 'sp500' covers the live date 2026-01-02 — list ends 2025-12-31
Note: the options layer ends 2025-08-29, so options features are missing after that date.
Live feature row for 2026-01-02: 1,234 stocks


,median20_auction_volume,log_median20_auction_volume,n_obs_median20,sp_quarterly_rebal,russell_recon,msci_review,any_index_event,monthly_opex_ex_quarterly,is_last_trading_day_month,is_last_trading_day_quarter,is_last_trading_day_year,is_first_trading_day_month,is_early_close,fomc_announcement_day,sp500_member_lag1,sp500_member_x_sp_rebal,sp500_add_pending,sp500_del_pending,sp500_add_trade_day,sp500_del_trade_day,td_to_sp500_change,opt_oi_exp_today_share,opt_oi_exp_7d_share,opt_log_total_oi,opt_pc_oi_ratio,opt_has_options_lag1,opt_to_stock_vol_lag1,opt_iv_atm_30d_lag1,opt_abs_gex_to_dollar_volume_lag1,volatility20,abnormal_volume_lag1,size_pct_lag1,market_volatility20,market_abnormal_volume_lag1,high_vol_regime,high_volume_regime,index_event_x_high_vol,opex_x_high_vol,month_end_x_high_vol,russell_recon_x_small_cap
available_pct,99.8000,99.8000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,100.0000,99.9000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000,100.0000


## 14. End-to-end check — the E0 → E5 ladder

Fit the same sequence as the methodology slide on the table just built, to confirm the features load into a model and behave as presented.

- Target: `y_log_ratio`. Model: ordinary least squares with an intercept.
- **Expanding window**: for each test year 2020–2025, fit on all model-ready rows from 2015 up to the end of the previous year, then score the test year.
- Missing feature values are filled with the training-window median; continuous features are capped at the training 0.5th and 99.5th percentiles. Both are learned inside each training window only.
- Metric: pooled out-of-sample log RMSE, $\sqrt{\text{mean}\big((y - \hat{y})^2\big)}$.

This is a check on the feature table, not a tuned model. The `midterm_slide` column shows the presented numbers for comparison: E0–E4 should be close;
E5 here also includes stock-level volatility, volume and size, so its gain is larger than on the slide.

In [17]:
LOG_FEATURES = ["volatility20", "abnormal_volume_lag1", "market_volatility20", "market_abnormal_volume_lag1"]
LADDER = [
    ("E0  historical reference + constant", []),
    ("E1  + index rebalancing events", ["sp_quarterly_rebal", "russell_recon", "msci_review"] + ([c for c in SP_FEATURES if c != "td_to_sp500_change"] if USE_SP500_STOCK_LEVEL else [])),
    ("E2  + monthly options expiration", ["monthly_opex_ex_quarterly"]),
    ("E3  + institutional calendar", ["is_last_trading_day_month", "is_last_trading_day_quarter", "is_last_trading_day_year", "is_first_trading_day_month", "is_early_close"]),
    ("E4  + FOMC meeting dates", ["fomc_announcement_day"]),
    ("E5  + volatility / volume regimes and interactions", E5_FEATURES),
]
if USE_OPTIONS:
    LADDER.append(("E5+ + stock-level options exposure (through Aug 2025)", OPT_FEATURES))

def fit_predict(train, test, cols):
    if not cols:
        return np.full(len(test), train["y_log_ratio"].mean())
    Xtr, Xte = train[cols].astype("float64").copy(), test[cols].astype("float64").copy()
    for c in cols:
        if c in LOG_FEATURES:
            Xtr[c], Xte[c] = np.log(Xtr[c].where(Xtr[c] > 0)), np.log(Xte[c].where(Xte[c] > 0))
        if Xtr[c].nunique() > 2:
            lo, hi = Xtr[c].quantile([0.005, 0.995])
            Xtr[c], Xte[c] = Xtr[c].clip(lo, hi), Xte[c].clip(lo, hi)
        fill = Xtr[c].median()
        fill = 0.0 if np.isnan(fill) else fill
        Xtr[c], Xte[c] = Xtr[c].fillna(fill), Xte[c].fillna(fill)
    A = np.column_stack([np.ones(len(Xtr)), Xtr.to_numpy()])
    beta, *_ = np.linalg.lstsq(A.T @ A, A.T @ train["y_log_ratio"].to_numpy(), rcond=None)      # normal equations
    return np.column_stack([np.ones(len(Xte)), Xte.to_numpy()]) @ beta

if RUN_SANITY_MODEL:
    ready = features[features.model_ready].assign(year=lambda d: d.date.dt.year)
    test_years = [y for y in range(2020, int(ready.year.max()) + 1)]
    errors = {name: [] for name, _ in LADDER}
    cols = []
    for name, add in LADDER:
        cols = cols + add
        for y in test_years:
            train, test = ready[ready.year < y], ready[ready.year == y]
            errors[name].append(test["y_log_ratio"].to_numpy() - fit_predict(train, test, cols))
    ladder = pd.DataFrame({"specification": list(errors),
                           "oos_log_rmse": [np.sqrt(np.mean(np.concatenate(e) ** 2)) for e in errors.values()]})
    presented = [0.6164, 0.540, 0.539, 0.529, 0.528, 0.5250]                      # midterm slide, for reference
    ladder["midterm_slide"] = presented + [np.nan] * (len(ladder) - len(presented))
    ladder["change_vs_previous"] = ladder["oos_log_rmse"].diff()
    ladder["improvement_vs_E0_pct"] = (1 - ladder["oos_log_rmse"] / ladder["oos_log_rmse"].iloc[0]) * 100
    n_test = int(ready.year.isin(test_years).sum())
    print(f"Out-of-sample stock-days ({test_years[0]}-{test_years[-1]}): {n_test:,}")
    ladder.to_csv(OUTPUT_DIR / "sanity_model_ladder.csv", index=False)
    check("Sanity model: adding economic features lowers out-of-sample error vs E0",
          ladder["oos_log_rmse"].iloc[min(5, len(ladder) - 1)] < ladder["oos_log_rmse"].iloc[0])
    display(ladder)

Out-of-sample stock-days (2020-2025): 1,882,373
[PASS] Sanity model: adding economic features lowers out-of-sample error vs E0


,specification,oos_log_rmse,midterm_slide,change_vs_previous,improvement_vs_E0_pct
0,E0 historical reference + constant,0.6163,0.6164,NaN,0.0000
1,E1 + index rebalancing events,0.5404,0.5400,-0.0759,12.3122
2,E2 + monthly options expiration,0.5400,0.5390,-0.0004,12.3810
3,E3 + institutional calendar,0.5307,0.5290,-0.0093,13.8953
4,E4 + FOMC meeting dates,0.5303,0.5280,-0.0004,13.9552
5,E5 + volatility / volume regimes and interact...,0.5142,0.5250,-0.0161,16.5667
6,E5+ + stock-level options exposure (through Au...,0.5141,NaN,-0.0002,16.5931


## 15. Quality-control report

In [18]:
qc = pd.DataFrame(QC)
qc.to_csv(OUTPUT_DIR / "qc_report.csv", index=False)
print(qc.status.value_counts().to_dict())
for status, label in (("FAIL", "FAILED CHECKS — review before using the features"), ("WARN", "WARNINGS — fine for the historical table; fix before live use")):
    if (qc.status == status).any():
        print(f"\n{label}:")
        display(qc[qc.status == status])
CON.close()
qc

{'PASS': 39, 'WARN': 1}

WARNINGS — fine for the historical table; fix before live use:


,check,status,detail
37,Reference list 'msci' covers the live date 202...,WARN,list ends 2025-11-24; extend reference_inputs ...


,check,status,detail
0,CRSP rows passing the universe filter,PASS,"4,927,078 of 4,927,078 rows on disk kept"
1,"CRSP (date, permno) unique",PASS,
2,"Auction (date, permno) unique",PASS,
3,CRSP daily volume non-negative,PASS,
4,Auction volume positive where present,PASS,
5,Auction price positive where present,PASS,
6,target_found agrees with auction_volume presence,PASS,
7,"Options (date, permno) unique",PASS,
8,Options volume / open interest non-negative,PASS,
9,Open interest expiring within 7 days never exc...,PASS,
